# End-to-End Sales Forecasting & Demand Intelligence System

This notebook completes the Week 3 and Week 4 internship project using the Superstore `train.csv` dataset and the supplementary `vgsales.csv` dataset.

Important honesty note: the local environment used to build this project did not have `statsmodels`, `prophet`, or `xgboost` installed. I wrote the notebook with the intended model imports and fallbacks so the work is reproducible after installing the packages, while still producing usable charts and business outputs from the available data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = Path("train.csv")
VG_PATH = Path("vgsales.csv")
df = pd.read_csv(DATA_PATH)
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")
vg = pd.read_csv(VG_PATH) if VG_PATH.exists() else None
df.head(), None if vg is None else vg.head()

## Task 1 - Data Loading, Merging & Deep Exploration

The dataset contains **9,800 rows**, **4,922 unique orders**, and covers **2015-01-03 to 2018-12-30**.

The supplementary `vgsales.csv` dataset is now included and merged at year level as external market context.

Merge pain point: the video game dataset does not share product IDs, customers, regions, or dates with the Superstore data. A row-level merge would be misleading, so I used a year-level merge for external demand context. This is useful for demonstrating multi-source handling, but it should not be treated as a causal driver of office/furniture/technology sales.

In [ ]:
def season(month):
    if month in (12, 1, 2): return "Winter"
    if month in (3, 4, 5): return "Spring"
    if month in (6, 7, 8): return "Summer"
    return "Autumn"

df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Week Number"] = df["Order Date"].dt.isocalendar().week.astype(int)
df["Day of Week"] = df["Order Date"].dt.day_name()
df["Quarter"] = df["Order Date"].dt.quarter
df["Season"] = df["Month"].apply(season)
df["Ship Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["MonthStart"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
df["WeekStart"] = df["Order Date"].dt.to_period("W").apply(lambda r: r.start_time)

missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()
weekly_sales = df.groupby("WeekStart")["Sales"].sum()
monthly_sales = df.groupby("MonthStart")["Sales"].sum().asfreq("MS").fillna(0)
missing.head(10), duplicates, weekly_sales.head(), monthly_sales.head()

In [ ]:
if vg is not None:
    vg["Year"] = pd.to_numeric(vg["Year"], errors="coerce")
    vg = vg.dropna(subset=["Year"])
    vg["Year"] = vg["Year"].astype(int)
    vg_yearly = vg.groupby("Year").agg(
        VG_Global_Sales_Millions=("Global_Sales", "sum"),
        VG_Title_Count=("Name", "count"),
        VG_Top_Genre=("Genre", lambda s: s.mode().iloc[0] if not s.mode().empty else "Unknown")
    ).reset_index()
    retail_yearly = df.groupby("Year")["Sales"].sum().reset_index(name="Superstore_Sales")
    merged_context = retail_yearly.merge(vg_yearly, on="Year", how="left")
    display(merged_context)
else:
    print("vgsales.csv not available")

### EDA Answers

- Highest revenue category: **Technology** with **$827,456** in sales.
- Most consistent regional growth: **East**, based on average yearly growth adjusted by volatility.
- Average order-to-ship time: **3.96 days**. Fastest region was **East (3.91 days)** and slowest was **Central (4.07 days)**.
- Consistent seasonal spike months by average monthly sales: **11, 12, 9**.

In [ ]:
df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
df.groupby(["Region", "Year"])["Sales"].sum().unstack()
df.groupby("Region")["Ship Days"].mean().sort_values()
df.groupby(["Year", "Month"])["Sales"].sum().reset_index().groupby("Month")["Sales"].mean().sort_values(ascending=False).head(12)

## Task 2 - Time Series Analysis & Decomposition

Because `statsmodels` was unavailable locally, I used a transparent manual decomposition: a 6-month centered rolling trend, average month seasonal index, and residual. The notebook below also shows the `statsmodels` version to use when the package is installed.

In [ ]:
monthly_sales.plot(figsize=(12, 4), title="Overall Monthly Sales Trend")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

In [ ]:
trend = monthly_sales.rolling(6, center=True, min_periods=3).mean()
seasonal = monthly_sales.groupby(monthly_sales.index.month).transform("mean") - monthly_sales.mean()
residual = monthly_sales - trend.bfill().ffill() - seasonal

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
axes[0].plot(monthly_sales.index, monthly_sales.values); axes[0].set_title("Observed")
axes[1].plot(trend.index, trend.values); axes[1].set_title("Trend")
axes[2].plot(seasonal.index, seasonal.values); axes[2].set_title("Seasonal")
axes[3].plot(residual.index, residual.values); axes[3].set_title("Residual")
plt.tight_layout()
plt.show()

In [ ]:
try:
    from statsmodels.tsa.stattools import adfuller
    result = adfuller(monthly_sales.dropna())
    print("ADF statistic:", result[0])
    print("p-value:", result[1])
    diff_result = adfuller(monthly_sales.diff().dropna())
    print("Differenced p-value:", diff_result[1])
except ModuleNotFoundError:
    print("statsmodels is not installed locally. Install it with: pip install statsmodels")
    print("Plain English: stationarity means the time series has a stable average and variance over time.")
    print("Visual result: this monthly sales series shows trend/seasonality, so differencing is likely needed before SARIMA.")

Observations:

- The series trends upward but with noisy month-to-month movement.
- Seasonality is visible, especially around late-year retail activity.
- Residual noise is highest around unusually large promotion-like spikes.
- The main pain point is that only 48 monthly points exist, which is not much data for robust monthly forecasting.

## Task 3 - Forecasting Using 3 Approaches

The ideal production notebook would use SARIMA, Prophet, and XGBoost. Locally, the heavy packages were missing, so the executed comparison uses:

- SARIMA fallback: seasonal naive forecast.
- Prophet fallback: linear trend plus month dummy regression.
- XGBoost fallback: RandomForest lag model.

This is not perfect, but it is honest and runnable.

In [ ]:
comparison = pd.read_csv("model_comparison.csv")
comparison

Recommended model from local metrics: **Prophet fallback - trend/month regression**. This recommendation is based on the lowest validation MAPE in the generated comparison table, not preference.

In [ ]:
# Ideal package imports for a fully installed environment:
# from statsmodels.tsa.statespace.sarimax import SARIMAX
# from prophet import Prophet
# from xgboost import XGBRegressor

# Local model comparison produced by the build script:
comparison[["Model", "MAE", "RMSE", "MAPE", "Forecast Month 1", "Forecast Month 2", "Forecast Month 3"]]

## Task 4 - Category and Region Forecasts

The best local model/fallback was repeated across Furniture, Technology, Office Supplies, West, and East. The chart is saved in `charts/segment_forecasts.png`.

In [ ]:
segment_forecasts = pd.read_csv("segment_forecasts.csv")
segment_forecasts

Strongest upcoming segment by the last forecast month: **Technology**.

## Task 5 - Anomaly Detection

Two methods were used: Isolation Forest and rolling Z-score. They do not always agree, which is useful: Isolation Forest looks at unusual patterns globally, while Z-score flags sharp deviations from a recent rolling baseline.

In [ ]:
anomalies = pd.read_csv("anomalies.csv")
anomalies[anomalies["IsAnomaly"] == True].head(20)

Top detected high-sales anomaly weeks:

- 2015-03-16: $37,704, likely promotion/festive demand or bulk ordering.
- 2018-11-26: $35,999, likely promotion/festive demand or bulk ordering.
- 2018-11-12: $30,572, likely promotion/festive demand or bulk ordering.

## Task 6 - Product Demand Segmentation

Sub-categories were clustered using total sales, growth rate, monthly volatility, and average order value. PCA was used only for 2D visualization.

In [ ]:
clusters = pd.read_csv("product_segments.csv")
clusters[["Sub-Category", "Total Sales", "YoY Growth Rate", "Monthly Volatility", "Average Order Value", "Demand Segment"]]

Stocking strategy:

- High Volume, Stable Core: keep higher safety stock and monitor service levels.
- Growing Demand: increase purchase planning gradually and review monthly.
- High Volatility Watchlist: avoid overcommitting inventory; use shorter replenishment cycles.
- Low Volume / Declining Demand: reduce bulk buying and consider clearance or made-to-order handling.

## Task 7 - Streamlit Dashboard

The dashboard is implemented in `app.py` with four pages:

- Sales Overview
- Forecast Explorer
- Anomaly Report
- Product Demand Segments

Run locally with:

```bash
streamlit run app.py
```

Deployment pain point: I cannot deploy to Streamlit Community Cloud from this local workspace without the user's GitHub/Streamlit account access. The app code and requirements are ready for deployment.

## Task 8 - Executive Report

The business report has been generated as `summary.pdf`.

## Process Pain Points

- Supplementary dataset is included now, but the join is only year-level because there is no common product/customer/store key.
- Forecasting packages were not installed locally, so fallbacks were needed.
- Only four years of monthly data gives roughly 48 points, which is small for confident monthly seasonality modeling.
- Superstore data still lacks stockouts, discounts, holidays, prices, and promotions.
- Dashboard deployment requires account-level access and cannot be completed only from local files.